In [6]:
from unsloth import is_bf16_supported, FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

import os
import json

from omegaconf import OmegaConf
from PIL import Image
from tqdm.notebook import tqdm

from beyond_hate.train.utils import binary_evaluation, convert_to_conversation_inference, extract_multi_labels, MultiVariableDataset, resize_and_pad

In [2]:
# System prompt for multi-variable classification
SYSTEM_TEXT = ("You are a content moderation assistant. You need to classify images with text "
               "across multiple dimensions: incivility and intolerance. "
               "- Incivility: Rude, disrespectful or dismissive tone towards others as well as opinions expressed with antinormative intensity."
               "- Intolerance: Behaviors that are threatening to democracy and pluralism - such as prejudice, segregation, hateful or violent speech, and the use of stereotyping in order to disqualify others and groups.")

USER_TEXT = ('Considering the image and its text: "{}". '
            'Classify this content on two dimensions: '
            '1. Incivility: Is this content civil or uncivil? '
            '2. Intolerance: Is this content tolerant or intolerant? '
            'Respond in the format: "Incivility: [Civil/Uncivil], Intolerance: [Tolerant/Intolerant]"')

In [3]:
cfg = OmegaConf.load("../../config/default.yaml")
eval_cfg = OmegaConf.load("../../config/eval.yaml")
# Merge training part with evaluation configuration
config = OmegaConf.merge(cfg.training, eval_cfg.training) 

In [4]:
# Load labeled data
with open(f'{cfg.out.path}/labels.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]
# Get relative paths for images
data = [{**item, 'img': '/'.join(item['img'].split('/')[-2:])} for item in data]
# Binarize labels
data = [{**item,
         'label_incivility': 1 if item['label_incivility'] > 0 else 0,
         'label_intolerance': 1 if item['label_incivility'] > 0 else 0}
         for item in data]
# Just keep items with valid image paths
data = [item for item in data if os.path.exists(f"{cfg.data.paths.hf}/{item['img']}")]

In [5]:
model, tokenizer = FastVisionModel.from_pretrained(
    config.model,
    load_in_4bit = config.load_in_4bit,
    use_gradient_checkpointing = config.use_gradient_checkpointing, 
    max_seq_length = config.max_seq_length
)

==((====))==  Unsloth 2025.6.4: Fast Clip patching. Transformers: 4.52.4.
   \\   /|    NVIDIA RTX A4500. Num GPUs = 1. Max memory: 19.599 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
# TODO: Batch processing
FastVisionModel.for_inference(model)

results = []
for sample in tqdm(data):
    label_intolerance = sample['label_intolerance']
    label_incivil = sample['label_incivility']
    label_hateful = sample['label_hateful']
    text = sample['text']
    
    # Resize and pad the image if specified in the config
    image = resize_and_pad(Image.open(f"{cfg.data.paths.hf}/{sample['img']}"), target_size=tuple(config.img_size), color=(255, 255, 255))
    
    conversation = convert_to_conversation_inference(text, SYSTEM_TEXT, USER_TEXT)
    prompt = tokenizer.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = tokenizer(images=image, text=prompt, return_tensors="pt").to("cuda:0")

    # autoregressively complete prompt
    max_new_tokens = 50
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    output = tokenizer.decode(output[0], skip_special_tokens=True)
    output = output.split('[/INST]')[-1]

    results.append(
        {
            'id': sample['id'],
            'label_intolerance': label_intolerance,
            'label_incivility': label_incivil,
            'label_hateful': label_hateful,
            'output': output,
        }
    )



  0%|          | 0/606 [00:00<?, ?it/s]

You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


In [8]:
# Extract true labels and predictions
y_true_incivil = [r['label_incivility'] for r in results]
y_true_intolerance = [r['label_intolerance'] for r in results]

y_pred = [extract_multi_labels(r['output']) for r in results]
y_pred_incivil = [pred[0] for pred in y_pred]
y_pred_intolerance = [pred[1] for pred in y_pred]

In [9]:
evaluation_incivil = binary_evaluation(y_true_incivil, y_pred_incivil)
evaluation_intolerance = binary_evaluation(y_true_intolerance, y_pred_intolerance)

In [5]:
import wandb

wandb.init(project=eval_cfg.wandb.project, name='baseline', config=dict(config), force=True)


wandb: Currently logged in as: nils_herrmann (nils_herrmann-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
wandb.log({
    'eval/incivil/accuracy': evaluation_incivil['accuracy'],
    'eval/incivil/precision': evaluation_incivil['precision'],
    'eval/incivil/recall': evaluation_incivil['recall'],
    'eval/incivil/f1': evaluation_incivil['f1_score'],
    'eval/incivil/confusion_matrix': wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_true_incivil,
        preds=y_pred_incivil,
        class_names=['Civil', 'Uncivil']
    ),
    'eval/intolerance/accuracy': evaluation_intolerance['accuracy'],
    'eval/intolerance/precision': evaluation_intolerance['precision'],
    'eval/intolerance/recall': evaluation_intolerance['recall'],
    'eval/intolerance/f1': evaluation_intolerance['f1_score'],
    'eval/intolerance/confusion_matrix': wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_true_intolerance,
        preds=y_pred_intolerance,
        class_names=['Tolerant', 'Intolerant']
    )
})

In [8]:
wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/incivil/accuracy,▁
eval/incivil/f1,▁
eval/incivil/precision,▁
eval/incivil/recall,▁
eval/intolerance/accuracy,▁
eval/intolerance/f1,▁
eval/intolerance/precision,▁
eval/intolerance/recall,▁
eval/incivil/accuracy,0.62099
eval/incivil/f1,0.6007
eval/incivil/precision,0.63122
